In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Load UCI Parkinson's Voice Dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"
df  = pd.read_csv(url)

print(f"Dataset shape:        {df.shape}")
print(f"Parkinson's rate:     {df['status'].mean()*100:.1f}%")
print(f"Missing values:       {df.isnull().sum().sum()}")
print(f"\nFeature categories:")
print(f"  MDVP voice features: {len([c for c in df.columns if 'MDVP' in c])}")
print(f"  Shimmer features:    {len([c for c in df.columns if 'Shimmer' in c])}")
print(f"  Other features:      {len([c for c in df.columns if 'MDVP' not in c and 'Shimmer' not in c])}")
print(f"\nAll columns:")
print(list(df.columns))
df.head(3)

Dataset shape:        (195, 24)
Parkinson's rate:     75.4%
Missing values:       0

Feature categories:
  MDVP voice features: 10
  Shimmer features:    5
  Other features:      11

All columns:
['name', 'MDVP:Fo(Hz)', 'MDVP:Fhi(Hz)', 'MDVP:Flo(Hz)', 'MDVP:Jitter(%)', 'MDVP:Jitter(Abs)', 'MDVP:RAP', 'MDVP:PPQ', 'Jitter:DDP', 'MDVP:Shimmer', 'MDVP:Shimmer(dB)', 'Shimmer:APQ3', 'Shimmer:APQ5', 'MDVP:APQ', 'Shimmer:DDA', 'NHR', 'HNR', 'status', 'RPDE', 'DFA', 'spread1', 'spread2', 'D2', 'PPE']


,name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE
0,phon_R01_S01_1,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,phon_R01_S01_2,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,phon_R01_S01_3,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634


In [3]:
# Cell 3 — Feature Engineering & Preprocessing
# Drop name column — identifier not a feature
df_clean = df.drop(columns=['name'])

# Key voice biomarkers for Parkinson's detection
# Jitter: frequency variation, Shimmer: amplitude variation
# HNR: harmonics-to-noise ratio, RPDE/DFA: nonlinear dynamics

X = df_clean.drop('status', axis=1)
y = df_clean['status']

# Engineer interaction features
X['Jitter_Shimmer']     = X['MDVP:Jitter(%)'] * X['MDVP:Shimmer']
X['HNR_NHR_ratio']      = X['HNR'] / (X['NHR'] + 1e-9)
X['Spread_interaction']  = X['spread1'] * X['spread2']
X['RPDE_DFA']           = X['RPDE'] * X['DFA']

print(f"Features after engineering: {X.shape[1]}")
print(f"Parkinson's cases:          {y.sum()} / {len(y)}")
print(f"Class balance:              {y.mean()*100:.1f}% positive")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler         = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

Features after engineering: 26
Parkinson's cases:          147 / 195
Class balance:              75.4% positive

Training samples: 156
Test samples:     39


In [4]:
# Cell 4 — Train 4 Models
# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42, eval_metric='logloss',
    verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc   = roc_auc_score(y_test, xgb_proba)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=6,
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)

# Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=3,
    learning_rate=0.05, random_state=42
)
gb_model.fit(X_train_scaled, y_train)
gb_proba = gb_model.predict_proba(X_test_scaled)[:, 1]
gb_auc   = roc_auc_score(y_test, gb_proba)

# SVM
svm_model = SVC(probability=True, kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)
svm_proba = svm_model.predict_proba(X_test_scaled)[:, 1]
svm_auc   = roc_auc_score(y_test, svm_proba)

print(f"{'Model':<25} {'ROC-AUC':>10}")
print("-" * 37)
print(f"{'XGBoost':<25} {xgb_auc:>10.4f}")
print(f"{'Random Forest':<25} {rf_auc:>10.4f}")
print(f"{'Gradient Boosting':<25} {gb_auc:>10.4f}")
print(f"{'SVM (RBF)':<25} {svm_auc:>10.4f}")

best_model = max([('XGBoost', xgb_auc), ('Random Forest', rf_auc),
                  ('Gradient Boosting', gb_auc), ('SVM', svm_auc)],
                  key=lambda x: x[1])
print(f"\nBest model: {best_model[0]} (AUC={best_model[1]:.4f})")

Model                        ROC-AUC
-------------------------------------
XGBoost                       0.9828
Random Forest                 0.9690
Gradient Boosting             0.9655
SVM (RBF)                     0.9828

Best model: SVM (AUC=0.9828)


In [7]:
# Cell 5 — ROC Curve & Feature Importance
from sklearn.metrics import roc_curve

fig = go.Figure()

models_plot = {
    'XGBoost':          (xgb_proba, 'crimson'),
    'Random Forest':    (rf_proba,  'royalblue'),
    'Gradient Boosting':(gb_proba,  'seagreen'),
    'SVM (RBF)':        (svm_proba, 'orange')
}

aucs = {
    'XGBoost': xgb_auc, 'Random Forest': rf_auc,
    'Gradient Boosting': gb_auc, 'SVM (RBF)': svm_auc
}

for name, (proba, color) in models_plot.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines',
        name=f'{name} (AUC={aucs[name]:.4f})',
        line=dict(color=color, width=2.5)
    ))

fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines',
    line=dict(color='gray', dash='dash'),
    name='Random Classifier'
))
fig.update_layout(
    title='ROC Curves — Parkinson\'s Tremor Detection',
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate (Sensitivity)',
    template='plotly_dark',
    width=800, height=500
)
fig.show()

# Feature importance from Random Forest
importance_df = pd.DataFrame({
    'feature':    X_train.columns.tolist(),
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(10)

fig2 = px.bar(
    importance_df, x='importance', y='feature',
    orientation='h',
    title='Top 10 Voice Biomarkers for Parkinson\'s Detection',
    template='plotly_dark',
    color='importance',
    color_continuous_scale='Oranges'
)
fig2.update_layout(width=800, height=450, yaxis=dict(autorange='reversed'))
fig2.show()

print("Top 10 voice biomarkers:")
print(importance_df.to_string(index=False))

Top 10 voice biomarkers:
      feature  importance
          PPE    0.116209
      spread1    0.111128
          NHR    0.054959
 MDVP:Flo(Hz)    0.054794
  MDVP:Fo(Hz)    0.054537
     MDVP:APQ    0.050876
 Shimmer:APQ5    0.046502
 MDVP:Fhi(Hz)    0.042884
     MDVP:RAP    0.042086
HNR_NHR_ratio    0.038958


In [8]:
# Cell 6 — Clinical Report & Export
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-24-parkinsons-tremor-detector\outputs'
import os
os.makedirs(output_dir, exist_ok=True)

# Confusion matrix for best model (XGBoost)
xgb_preds = xgb_model.predict(X_test)
cm = confusion_matrix(y_test, xgb_preds)
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ppv         = tp / (tp + fp)
npv         = tn / (tn + fn)

print("=== Clinical Performance — XGBoost ===")
print(f"\n  ROC-AUC:     {xgb_auc:.4f}")
print(f"  Sensitivity: {sensitivity:.4f}  (% Parkinson's correctly detected)")
print(f"  Specificity: {specificity:.4f}  (% healthy correctly identified)")
print(f"  PPV:         {ppv:.4f}  (precision when positive)")
print(f"  NPV:         {npv:.4f}  (precision when negative)")
print(f"\n  Confusion Matrix:")
print(f"  TP={tp}  FP={fp}")
print(f"  FN={fn}  TN={tn}")

print(f"\n=== Full Classification Report ===")
print(classification_report(y_test, xgb_preds,
      target_names=['Healthy', "Parkinson's"]))

# Export
importance_df.to_csv(f'{output_dir}\\feature_importance.csv', index=False)
pd.DataFrame({
    'model':   ['XGBoost', 'Random Forest', 'Gradient Boosting', 'SVM'],
    'roc_auc': [xgb_auc, rf_auc, gb_auc, svm_auc]
}).to_csv(f'{output_dir}\\model_results.csv', index=False)

# Risk scores
df_results = X_test.copy()
df_results['risk_score']  = xgb_proba
df_results['prediction']  = xgb_preds
df_results['actual']      = y_test.values
df_results.to_csv(f'{output_dir}\\predictions.csv', index=False)

print(f"\nExports saved to outputs/")

=== Clinical Performance — XGBoost ===

  ROC-AUC:     0.9828
  Sensitivity: 0.9655  (% Parkinson's correctly detected)
  Specificity: 0.8000  (% healthy correctly identified)
  PPV:         0.9333  (precision when positive)
  NPV:         0.8889  (precision when negative)

  Confusion Matrix:
  TP=28  FP=2
  FN=1  TN=8

=== Full Classification Report ===
              precision    recall  f1-score   support

     Healthy       0.89      0.80      0.84        10
 Parkinson's       0.93      0.97      0.95        29

    accuracy                           0.92        39
   macro avg       0.91      0.88      0.90        39
weighted avg       0.92      0.92      0.92        39


Exports saved to outputs/
